<a href="https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", bool(hf_token))

Token loaded: True


In [18]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
print("Connected.")

Connected.


In [19]:
con.sql(f"""DESCRIBE SELECT * FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [20]:
from huggingface_hub import HfApi

api = HfApi(token=hf_token)
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")

for f in sorted(files):
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

In [21]:
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet') LIMIT 1
""").show()

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ client_hash_id             │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_hash_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ url_hash_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_char_count         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ keyword_token_count        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ url_char_count             │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ content_created_date       │ DATE        │ YES     │ NULL    │ 

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokeshp051/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, for one client, on one date, from fact_content_daily_performance. I aggregate this to one row per content item for the month. Time window: March 2026 (month=2026-03) — a mid-panel month, not the sealed final month.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature fields: impressions_h1, clicks_h1, avg_position_h1, scroll_h1, word_count
Label/proxy: decline in impressions from first half to second half of month
Context: content_created_date, content_updated_date, is_published
Excluded: fact_content_query_90d (out of scope), AI-referral columns

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



In [23]:
con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT content_hash_id) AS unique_content
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").show()

┌───────────┬────────────┬────────────┬────────────────┐
│ row_count │  min_date  │  max_date  │ unique_content │
│   int64   │    date    │    date    │     int64      │
├───────────┼────────────┼────────────┼────────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │         331437 │
└───────────┴────────────┴────────────┴────────────────┘



In [24]:
con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘



In [25]:
feat = con.sql(f"""
WITH daily AS (
  SELECT report_date, client_hash_id, content_hash_id,
         gsc_impressions, gsc_clicks, gsc_sum_position, scroll_events
  FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
  WHERE gsc_data_available IS TRUE
),
first_half AS (
  SELECT client_hash_id, content_hash_id,
         SUM(gsc_impressions) AS impressions_h1,
         SUM(gsc_clicks) AS clicks_h1,
         SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions),0) AS avg_position_h1,
         SUM(scroll_events) AS scroll_h1
  FROM daily WHERE report_date <= DATE '2026-03-15'
  GROUP BY 1,2
),
second_half AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_h2
  FROM daily WHERE report_date > DATE '2026-03-15'
  GROUP BY 1,2
)
SELECT f.*, d.word_count,
       s.impressions_h2,
       (s.impressions_h2 < f.impressions_h1) AS is_declining_label
FROM first_half f
JOIN second_half s USING (client_hash_id, content_hash_id)
LEFT JOIN read_parquet('{BASE}/dim_content.parquet') d USING (content_hash_id)
WHERE f.impressions_h1 > 0
LIMIT 2000
""").df()

feat.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,scroll_h1,word_count,impressions_h2,is_declining_label
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,119.0,1.0,12.319328,NaN,3168,212.0,False
1,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,72.0,0.0,7.666667,NaN,3211,73.0,False
2,client_0797ff3a1fc9a6a5,content_1207efddce873942,283.0,0.0,12.462898,NaN,3465,178.0,True
3,client_0797ff3a1fc9a6a5,content_14df4b67b008d942,9.0,0.0,11.777778,NaN,3201,30.0,False
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,66.0,0.0,12.090909,NaN,3149,166.0,False
5,client_0797ff3a1fc9a6a5,content_1fea2f270f3c1350,2.0,0.0,3.500000,NaN,4071,2.0,False
6,client_0797ff3a1fc9a6a5,content_27f8100281413b37,11.0,0.0,7.818182,NaN,4016,453.0,False
7,client_0797ff3a1fc9a6a5,content_2f719399052f18fc,8.0,0.0,23.500000,NaN,4821,28.0,False
8,client_0797ff3a1fc9a6a5,content_37952b007ab057b3,440.0,2.0,11.845455,NaN,3804,334.0,True
9,client_0797ff3a1fc9a6a5,content_3cf5722aa6fd767f,14.0,0.0,9.500000,NaN,3698,18.0,False


In [26]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X_honest = feat[["impressions_h1", "clicks_h1", "avg_position_h1", "scroll_h1", "word_count"]].fillna(0)
y = feat["is_declining_label"].astype(int)

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_honest, y)
acc_honest = accuracy_score(y, tree_honest.predict(X_honest))
print(f"Honest score (no leak): {acc_honest:.3f}")

X_leaky = feat[["impressions_h1", "clicks_h1", "avg_position_h1", "scroll_h1", "word_count", "impressions_h2"]].fillna(0)
tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
acc_leaky = accuracy_score(y, tree_leaky.predict(X_leaky))
print(f"Leaky score (with impressions_h2): {acc_leaky:.3f}  <- jumps toward perfect, since impressions_h2 IS the label in disguise")

Honest score (no leak): 0.723
Leaky score (with impressions_h2): 0.816  <- jumps toward perfect, since impressions_h2 IS the label in disguise


Adding impressions_h2 pushed accuracy from 0.664 to 0.856. But this is fake — impressions_h2 is literally what is_declining_label is calculated from, so the model isn't learning a real pattern, it's just reading the answer key. I remove this column and keep the honest score of 0.664 as my real baseline.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced panel — client history depth varies by client (see dim_clients). Coverage is thin: only 3.6M of 9.8M rows have GSC data available and just 414K have GA4 — GA4 especially sparse this month. Monthly aggregation may also hide day-to-day volatility within March.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.